# Shell-model interaction: ML ceiling, ab-initio baseline, and spectra

End-to-end reproducible pipeline. Imports the four modules in `src/`:
`interactions.py`, `features.py`, `symbolic.py`, `spectrum.py`.

It reproduces the project's verified numbers:
1. **natural floor** RMS(USDA, USDB) = 0.267 MeV
2. **baseline scan** — modern ab-initio (JISP/N3LO) predicts USDB ~0.47 MeV with zero fit
3. **monopole correction** — JISP + 2 per-T constants -> ~0.40 MeV
4. **ML ceiling** — symbolic regression on quantum features -> CV ~1.4 MeV
5. **spectra** — USDB & JISP reproduce 18O / 18F levels (validates the application)

> Kaggle: Internet = ON. Put the `src/` folder next to this notebook (or upload
> the four .py files). PySR installs Julia on first import (a few minutes).

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pysr"], check=False)
# make src/ importable (works for GitHub layout notebooks/.. and for flat Kaggle uploads)
for p in ["src", "../src", ".", "/kaggle/working/src", "/kaggle/working"]:
    if os.path.isdir(p) or os.path.isfile(os.path.join(p, "interactions.py")):
        sys.path.insert(0, p)
import numpy as np, pandas as pd
import interactions as I
from features import build_features, QUANTUM_FEATURES, PHYSICS_FEATURES
import spectrum as S
print("modules loaded:", I.__name__, "features", "spectrum")

## 1. Load interactions + natural floor

In [ ]:
usdb, usda = I.load("USDB"), I.load("USDA")
a, b = usdb.canonical_tbme(), usda.canonical_tbme()
keys = [k for k in a if k in b]
floor = np.sqrt(np.mean([(a[k] - b[k])**2 for k in keys]))
print(f"USDB SPE: {{o: round(usdb.spe[i],3) for i,o in [(1,'d3/2'),(2,'d5/2'),(3,'s1/2')]}}")
print(f"natural floor RMS(USDA,USDB) = {floor:.3f} MeV   (expect 0.267)")

## 2. Baseline scan — which interaction predicts USDB with zero fitting?

In [ ]:
target = usdb.canonical_tbme()
print(f"{'baseline':16s} {'RMS->USDB':>10s}")
scan = {}
for name in ["USDA", "KUO", "WILDENTHAL", "N3LO", "JISP"]:
    m = I.load(name).canonical_tbme()
    ks = [k for k in target if k in m]
    rms = np.sqrt(np.mean([(target[k] - m[k])**2 for k in ks]))
    scan[name] = rms
    print(f"{name:16s} {rms:>10.3f}")
print("\n=> modern ab-initio (N3LO/JISP) ~0.47, far below the ~1.1-1.4 ML ceiling.")

## 3. Monopole correction (JISP + 2 per-T constants)

In [ ]:
jisp = I.load("JISP")
cU, cB = usdb.monopole(), jisp.monopole()
byT = {0: [], 1: []}
for k in cU:
    if k in cB: byT[k[2]].append(cU[k] - cB[k])
dT = {T: float(np.mean(v)) for T, v in byT.items()}
tj, tu = jisp.canonical_tbme(), usdb.canonical_tbme()
ks = [k for k in tu if k in tj]
def rms(pred): return np.sqrt(np.mean([(tu[k] - pred(k))**2 for k in ks]))
bare = rms(lambda k: tj[k])
corr = rms(lambda k: tj[k] + (dT[k[5]] if (k[0],k[1])==(k[2],k[3]) else 0.0))
print(f"JISP bare              RMS->USDB = {bare:.3f} MeV")
print(f"JISP + monopole(2-par) RMS->USDB = {corr:.3f} MeV")
print(f"per-T shifts: T0={dT[0]:+.3f}  T1={dT[1]:+.3f} MeV   (T0 transfers to fp)")

## 4. ML ceiling — symbolic regression on quantum-number features

In [ ]:
from symbolic import fit_sr, cv_rmse, best_equations
df = build_features(usdb)
model, ins = fit_sr(df, PHYSICS_FEATURES)
print(f"in-sample RMSE = {ins:.3f} MeV")
print(best_equations(model).to_string(index=False))
mu, sd = cv_rmse(df, PHYSICS_FEATURES)
print(f"CV-RMSE = {mu:.3f} +/- {sd:.3f} MeV   (the information ceiling ~1.4)")

## 5. Spectra — validate the application on real nuclei (18O, 18F)

In [ ]:
def show(label, lv, cut, fmt):
    print(label)
    seen = set()
    for ex, J, T in lv:
        if ex <= cut and (J, T, ex) not in seen:
            seen.add((J, T, ex)); print("   " + fmt(J, T, ex))

print("== 18O (exp: 2+ 1.98, 4+ 3.55) ==")
_, lv = S.o18(usdb);                 show("USDB", lv, 4.0, lambda J,T,e: f"{J}+  E*={e:.3f}")
_, lv = S.o18(jisp, spe=usdb.spe);   show("JISP (USDB SPE)", lv, 4.0, lambda J,T,e: f"{J}+  E*={e:.3f}")

print("\n== 18F (exp: 1+ gs, 3+ 0.94, 0+(T1) 1.04) ==")
_, lv = S.f18(usdb);                 show("USDB", lv, 1.6, lambda J,T,e: f"J={J} T={T}  E*={e:.3f}")
_, lv = S.f18(jisp, spe=usdb.spe);   show("JISP (USDB SPE)", lv, 1.6, lambda J,T,e: f"J={J} T={T}  E*={e:.3f}")

## Summary of verified numbers

| quantity | value |
|---|---|
| natural floor RMS(USDA,USDB) | 0.267 MeV |
| best ab-initio baseline (JISP) | ~0.47 MeV (0 params) |
| JISP + 2-param monopole | ~0.40 MeV |
| ML ceiling (SR, CV) | ~1.4 MeV |
| USDB 18O 2+ / 18F 1+ gs | 1.998 MeV / correct |

**Story:** ML measures the information ceiling of quantum-number features;
modern ab-initio supplies the missing physics (reproduces sd spectra unfitted);
the monopole correction's isoscalar part is transferable but does not by itself
improve spectroscopy — a cautionary, honest result.